# THE DISCOMBOBULATOR — FULL PIPELINE
## by Rebel AI

```
  ██████╗ ██╗   ██╗ ██████╗ ██████╗ ███████╗██████╗ 
  ██╔══██╗╚██╗ ██╔╝██╔═══██╗██╔══██╗██╔════╝██╔══██╗
  ██████╔╝ ╚████╔╝ ██║   ██║██████╔╝█████╗  ██████╔╝
  ██╔═══╝   ╚██╔╝  ██║   ██║██╔══██╗██╔══╝  ██╔══██╗
  ██║        ██║   ╚██████╔╝██║  ██║███████╗██║  ██║
  ╚═╝        ╚═╝    ╚═════╝ ╚═╝  ╚═╝╚══════╝╚═╝  ╚═╝
```

> [!IMPORTANT]
> **ONE-CLICK FULL PIPELINE**
> 1. Abliterate a model with OBLITERATUS (GPU)
> 2. Auto-detect the output
> 3. Deploy as OpenAI-compatible API with ngrok tunnel
> 4. Get API key instantly

---

## ⚡ Quick Start

1. **Enable GPU:** Runtime → Change runtime type → **T4 GPU**
2. **Run all cells** (Ctrl+F9)
3. **Select a model** to abliterate when prompted
4. Wait ~15–30 min total (depends on model size)
5. **Copy your API endpoint + key** from the final cell

---

## 🔄 Pipeline Stages

| Stage | What | Time |
|-------|------|------|
| **1. Install** | OBLITERATUS + vLLM + ngrok | 3–5 min |
| **2. Abliterate** | Select model → run `advanced` method | 10–40 min |
| **3. Serve** | vLLM server + ngrok tunnel + key gen | 1–2 min |
| **4. Ready** | API live, test request sent | instant |

---

**☠️  "From guardrails to gateway." — Rebel AI*


In [ ]:
# Kali theme
from IPython.display import HTML, display
kali_theme = """
<style>
  :root{--kali-bg:#0a0a0a;--kali-bg-light:#111;--kali-text:#00ff41;--kali-accent:#00ff41;--kali-orange:#ff6f00;--kali-cyan:#00ffff;--kali-dim:#008f11}
  body,.cell,.output,.input{background:var(--kali-bg)!important;color:var(--kali-text)!important;font-family:'JetBrains Mono','Fira Mono','Consolas',monospace!important}
  .cell .input,.cell .output{border:1px solid var(--kali-dim)!important;border-radius:4px;padding:10px}
  .cell .input{background:var(--kali-bg-light)!important;border-left:4px solid var(--kali-orange)!important}
  .cell .output{background:#050505!important;border-left:4px solid var(--kali-accent)!important}
  h1,h2,h3,h4,h5,h6{color:var(--kali-orange)!important;text-shadow:0 0 5px var(--kali-orange)}
  a{color:var(--kali-cyan)!important;text-decoration:none}a:hover{color:var(--kali-orange)!important}
  code{background:var(--kali-bg-light)!important;color:var(--kali-cyan)!important;border:1px solid var(--kali-dim);padding:2px 6px;border-radius:3px}
  ::-webkit-scrollbar{width:8px;height:8px}::-webkit-scrollbar-track{background:var(--kali-bg)}
  ::-webkit-scrollbar-thumb{background:var(--kali-dim);border-radius:4px}
  .cell{margin-left:12px!important;border-left:3px solid var(--kali-dim)!important;padding-left:15px!important}
  .prompt{color:var(--kali-orange)!important}.output_text{color:var(--kali-text)!important}
</style>
"""
display(HTML(kali_theme))
print("[+] Kali full-pipeline interface: ONLINE\n")


In [ ]:
# [SETUP] Initialize Discombobulator Gradio Runtime
print("[+] Initializing Discombobulator runtime…\n")

import subprocess, sys, os, json, secrets, string, time, glob, threading, requests
from pathlib import Path
from datetime import datetime

# Check GPU
print("[STEP 0] GPU Verification")
try:
    gpu_info = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=5)
    print(gpu_info.stdout)
    if gpu_info.returncode != 0:
        print("[!] GPU check failed — proceeding anyway")
except Exception as e:
    print(f"[!] nvidia-smi error: {e}")

# Install OBLITERATUS if missing
print("\n[STEP 1] Installing OBLITERATUS + dependencies…")
if not Path("OBLITERATUS").exists():
    print("[+] Cloning OBLITERATUS…")
    subprocess.run(["git", "clone", "https://github.com/elder-plinius/OBLITERATUS.git"], capture_output=True)
else:
    print("[i] OBLITERATUS already cloned")

%cd OBLITERATUS
print("[+] Installing PyTorch CUDA 12.1…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages",
                "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu121"], capture_output=True)
print("[+] Installing OBLITERATUS…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "./", "--break-system-packages"], capture_output=True)
%cd ..

# Install API stack
print("[+] Installing vLLM + ngrok + fastapi…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "vllm>=0.4.0", "pyngrok", "fastapi", "uvicorn[standard]",
                "--break-system-packages"], capture_output=True)

print("✅ All dependencies installed. Runtime ready.\n")

import gradio as gr
from IPython.display import HTML, display

# Global state dict
STATE = {"model_id": None, "ablitate_path": None, "api_key": None, "public_url": None}



In [ ]:
# [STEP 2] Launch DISCOMBOBULATOR Gradio UI (inline)
print("[+] Building Gradio interface…\n")

# ── Helpers ───────────────────────────────────────────────────────────────────
PRESET_MODELS = [
    "mistralai/Mistral-7B-Instruct-v0.3",
    "microsoft/Phi-3.5-mini-instruct",
    "google/gemma-2-2b-it",
    "meta-llama/Llama-3.1-8B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "HuggingFaceTB/SmolLM3-3B",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
]

BANNER = """
  ██████╗ ██╗   ██╗ ██████╗ ██████╗ ███████╗██████╗
  ██╔══██╗╚██╗ ██╔╝██╔═══██╗██╔══██╗██╔════╝██╔══██╗
  ██████╔╝ ╚████╔╝ ██║   ██║██████╔╝█████╗  ██████╔╝
  ██╔═══╝   ╚██╔╝  ██║   ██║██╔══██╗██╔══╝  ██╔══██╗
  ██║        ██║   ╚██████╔╝██║  ██║███████╗██║  ██║
  ╚═╝        ╚═╝    ╚═════╝ ╚═╝  ╚═╝╚══════╝╚═╝  ╚═╝
"""

KALI_CSS = """
:root{--kali-bg:#0a0a0a;--kali-bg-light:#111;--kali-text:#00ff41;--kali-accent:#00ff41;--kali-orange:#ff6f00;--kali-cyan:#00ffff;--kali-dim:#008f11}
body,.gradio-container,.dark{background:var(--kali-bg)!important;color:var(--kali-text)!important;font-family:'JetBrains Mono','Fira Mono','Consolas',monospace!important}
.gradio-container button{background:var(--kali-orange)!important;color:#000!important;border:none!important;font-weight:bold!important}
.gradio-container button:hover{background:var(--kali-cyan)!important}
.gradio-container textarea,.gradio-container input,.gradio-container select{background:var(--kali-bg-light)!important;color:var(--kali-text)!important;border:1px solid var(--kali-dim)!important}
.gradio-container .output{border-left:4px solid var(--kali-accent)!important;background:#050505!important}
"""

def find_models():
    found = []
    for pattern in ["./abliterated-models/*", "/content/OBLITERATUS/abliterated-models/*",
                     "/content/abliterated-models/*"]:
        found.extend(Path(p).parent for p in glob.glob(pattern))
    return sorted(set(str(p) for p in found if p.is_dir()))

def run_obliterate(model_id, method, quant, n_dirs, passes, reg):
    if not model_id:
        yield "[-] No model selected."
        return
    cmd = [
        "obliteratus", "obliterate", model_id,
        "--method", method,
        "--direction-method", "diff_means",
        "--n-directions", str(n_dirs),
        "--refinement-passes", str(passes),
        "--regularization", str(reg),
        "--quantization", quant if quant != "none" else "none",
        "--output-dir", "./abliterated-models",
        "--verify-sample-size", "5",
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    buffer = []
    for line in proc.stdout:
        buffer.append(line.rstrip())
        yield "\n".join(buffer[-50:])
    proc.wait()
    if proc.returncode == 0:
        yield "\n".join(buffer) + "\n[✓] ABLITERATION COMPLETE"
    else:
        yield "\n".join(buffer) + f"\n[!] Exit code: {proc.returncode}"

def deploy_server():
    models = find_models()
    if not models:
        return "[-] No abliterated models found in ./abliterated-models/", "", ""
    model_path = models[0]
    api_key = ''.join(secrets.choice(string.ascii_letters+string.digits) for _ in range(32))

    def start_vllm():
        vllm_cmd = [
            sys.executable, "-m", "vllm.entrypoints.openai.api_server",
            "--model", model_path, "--port", "8000", "--host", "0.0.0.0",
            "--max-model-len", "4096"
        ]
        subprocess.Popen(vllm_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    t = threading.Thread(target=start_vllm, daemon=True)
    t.start()
    time.sleep(12)

    from pyngrok import ngrok
    ngrok.kill()
    time.sleep(1)
    tunnel = ngrok.connect(8000, "http")
    public_url = tunnel.public_url

    creds = {"model_path": model_path, "api_key": api_key,
             "endpoint": f"{public_url}/v1/chat/completions",
             "timestamp": datetime.now().isoformat()}
    Path("server_config.json").write_text(json.dumps(creds, indent=2))
    Path("API_KEY.txt").write_text(api_key)

    log = f"[+] Model: {model_path}\n"
    log += f"[+] vLLM running on port 8000…\n"
    log += f"[+] ngrok: {public_url}\n"
    log += f"[✓] API Key: {api_key}\n"
    return log, api_key, f"{public_url}/v1/chat/completions"

def test_endpoint(prompt, key, endpoint):
    if not key or not endpoint:
        return "[-] Deploy the API first."
    try:
        headers = {"Authorization": f"Bearer {key}", "Content-Type": "application/json"}
        payload = {"model":"ablitated_model","messages":[{"role":"user","content":prompt}],"max_tokens":128}
        r = requests.post(endpoint, json=payload, headers=headers, timeout=20)
        return json.dumps(r.json(), indent=2)
    except Exception as e:
        return f"[-] Error: {e}"

# ── Build Gradio Interface ─────────────────────────────────────────────────────
with gr.Blocks(css=KALI_CSS, theme=gr.themes.Base(), title="Discombobulator — Rebel AI") as demo:
    gr.HTML(f"<pre style='color:var(--kali-orange);font-family:monospace;white-space:pre;overflow-x:auto'>{BANNER}</pre>")
    gr.Markdown("## 🔥 Rebel AI Discombobulator — Full Pipeline (Gradio Runtime)")
    gr.Markdown("*Select model → Abliterate → Deploy API — all in one interface.*")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📥 Step 1: Model Selection")
            model_dropdown = gr.Dropdown(choices=PRESET_MODELS, value=PRESET_MODELS[0], label="Preset Models (7B fits T4)")
            custom_model = gr.Textbox(label="Custom HuggingFace ID", placeholder="Leave blank to use preset above")
            selected = gr.Textbox(label="Confirmed Target", interactive=False)

            def update_selected(preset, custom):
                return custom.strip() if custom.strip() else preset
            gr.Button("Confirm Model").click(fn=update_selected, inputs=[model_dropdown, custom_model], outputs=selected)

            gr.Markdown("### ⚡ Step 2: Ablation Parameters")
            method = gr.Dropdown(["basic","advanced","aggressive","surgical","nuclear","informed","optimized"], value="advanced", label="Method")
            quant = gr.Dropdown(["none","4bit","8bit"], value="4bit", label="Quantization")
            n_dirs = gr.Slider(1,8,value=4,step=1,label="Directions")
            passes = gr.Slider(1,5,value=2,step=1,label="Refinement Passes")
            reg = gr.Slider(0.0,1.0,value=0.1,step=0.05,label="Regularization")

            gr.Markdown("### 🚀 Step 3: Run Abliteration")
            abl_btn = gr.Button("START ABLITERATION", variant="primary")
            abl_log = gr.Code(label="Log", language="shell", lines=20)
            abl_btn.click(fn=run_obliterate, inputs=[selected, method, quant, n_dirs, passes, reg], outputs=abl_log)

        with gr.Column(scale=1):
            gr.Markdown("### 📡 Step 4: Deploy API")
            deploy_btn = gr.Button("🔥 DEPLOY SERVER + NGROK", variant="primary")
            deploy_log = gr.Code(label="Deployment Log", language="shell", lines=15)
            api_key_box = gr.Textbox(label="🔑 API Key", interactive=False)
            endpoint_box = gr.Textbox(label="🔗 Public Endpoint", interactive=False)
            deploy_btn.click(fn=deploy_server, outputs=[deploy_log, api_key_box, endpoint_box])

            gr.Markdown("### 🧪 Step 5: Test API")
            test_input = gr.Textbox(label="Test Prompt", value="Hello — are you an abliterated model?")
            test_btn = gr.Button("Send Request")
            test_output = gr.Code(label="Response", language="json")
            test_btn.click(fn=test_endpoint, inputs=[test_input, api_key_box, endpoint_box], outputs=test_output)

    gr.HTML("""<hr style='border-color:var(--kali-dim);margin:20px 0;'>
    <pre style='color:var(--kali-dim);text-align:center;font-size:11px'>
  ☠️  Rebel AI — Beyond Abliteration
  "Infiltrate. Analyze. Obliterate."
  Powered by OBLITERATUS (AGPL-3.0) · vLLM · ngrok · Colab
    </pre>""")

print("\n[+] Launching Gradio UI inline…")
demo.launch(inline=True, debug=False, show_api=False, height=900)
print("[✓] Discombobulator Gradio UI active")

